## Marine Microbe database
run sequence data through a marine microbes focussed database to compare to kraken2

MARMICRODB database for taxonomic classification of (marine) metagenomes: https://zenodo.org/records/10999524

use with Kaiju: https://github.com/bioinformatics-centre/kaiju

In [ ]:
#downloading database to /scratch4/workspace/nikea_ulrich_uml_edu-col_data/marmicrodb
#tmux session db-download on login1

wget -O MARMICRODB.fmi https://zenodo.org/records/10999524/files/MARMICRODB.fmi?download=1
wget -O names.dmp https://zenodo.org/records/10999524/files/names.dmp?download=1
wget -O nodes.dmp https://zenodo.org/records/10999524/files/nodes.dmp?download=1
wget -O README.md https://zenodo.org/records/10999524/files/README.md?download=1

In [ ]:
##INSTALLATION 
module load conda/latest
conda create --prefix /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kaiju
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kaiju
conda install -c bioconda kaiju

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --qos=long # if needs longer
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/slurms/slurm-kaiju_marinedb-ofav_try-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kaiju

#parameters
DB="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/marmicrodb"
READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/012025_final_filtered/repaired"
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/marmicrodb/kaiju_try"
mkdir -p $OUTDIR

#let's try with 1 ofav sample first before running all
kaiju -z 20 -a greedy -e 5 -m 11 -s 65 -E 0.05 -x \
-t $DB/nodes.dmp -f $DB/MARMICRODB.fmi \
-i $READS/012025_COL_SAN_T5_568_OFAV_S1_host_removed_R1.tagged_filter_ready.fastq.gz \
-j $READS/012025_COL_SAN_T5_568_OFAV_S1_host_removed_R2.tagged_filter_ready.fastq.gz \
-o $OUTDIR/012025_COL_SAN_T5_568_OFAV_S1.kaiju

conda deactivate

# JOB-ID:55527494
# bash script file name: /bash_scripts/042026_analysis/kaiju_marinedb.sh

In [ ]:
#turn output into summary table (this is similar to kraken output)
kaiju2table -t nodes.dmp -n names.dmp -r species -o kaiju_try/012025_COL_SAN_T5_568_OFAV_S1_kaiju_S.tsv kaiju_try/012025_COL_SAN_T5_568_OFAV_S1.kaiju

#well this didn't help: 99.136287% reads unclassified

I think I will run it on all the ofavs and create a taxonomy plot to compare to the kraken output and see what Sarah thinks.

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=180G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --qos=long # if needs longer
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/slurms/slurm-kaiju_marinedb-ofav_012025-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kaiju

#parameters
DB="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/marmicrodb"
READS="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/012025_final_filtered/repaired"
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/marmicrodb/ofav_kaiju_output"
mkdir -p $OUTDIR
LISTPATH="/work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/ID_tables"
SAMPLELIST="012025_ofav_sampleids.txt"

#run repaired reads through Kaiju with Marine microbes database
while IFS= read -r SAMPLEID; do
kaiju -z 20 -a greedy -e 5 -m 11 -s 65 -E 0.05 -x \
-t $DB/nodes.dmp -f $DB/MARMICRODB.fmi \
-i $READS/"${SAMPLEID}"_host_removed_R1.tagged_filter_ready.fastq.gz \
-j $READS/"${SAMPLEID}"_host_removed_R2.tagged_filter_ready.fastq.gz \
-o $OUTDIR/"${SAMPLEID}".kaiju
if [ $? -eq 0 ]; then
        echo "kaiju completed successfully for sample: $SAMPLEID"
    else
        echo "kaiju encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kaiju: All samples processed successfully."

conda deactivate

# JOB-ID:55532797
# bash script file name: /bash_scripts/042026_analysis/kaiju_marinedb_ofav_012025.sh

and ran for 032024 samples too: 55532800

resource for turning the summary tables into ASV-like tables: https://github.com/bioinformatics-centre/kaiju/issues/17 or https://github.com/bioinformatics-centre/kaiju/issues/36 \
need to make sure I include the whole taxa string

this might be it: https://github.com/gisleDK/Biopythonpieces/blob/master/Scripts/kaiju2phyloseq.py

maybe do this too? The program kaiju2krona can be used to convert Kaiju's tab-separated output file into a tab-separated text file, which can be imported into Krona. It requires the nodes.dmp and names.dmp files from the NCBI taxonomy for mapping the taxon identifiers from Kaiju's output to the corresponding taxon names. https://github.com/bioinformatics-centre/kaiju#creating-classification-summary

#### make summary lists for each individual kaiju output

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=10G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 24:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/slurms/slurm-kaiju2table-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/.conda/envs/kaiju

#parameters
DB="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/marmicrodb"
OUTDIR="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/marmicrodb/ofav_kaiju_output"
LISTPATH="/scratch4/workspace/nikea_ulrich_uml_edu-col_data/marmicrodb/ofav_kaiju_output"
SAMPLELIST="ofav_samples.txt"

#turn Kaiju outputs into a readable summary table
while IFS= read -r SAMPLEID; do
kaiju2table -t $DB/nodes.dmp -n $DB/names.dmp -r species -o $OUTDIR/"${SAMPLEID}"_S.tsv $OUTDIR/"${SAMPLEID}".kaiju
if [ $? -eq 0 ]; then
        echo "kaiju2 table completed successfully for sample: $SAMPLEID"
    else
        echo "kaiju2table encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kaiju2table: All samples processed successfully."

conda deactivate

# JOB-ID:55555575
# bash script file name: /bash_scripts/042026_analysis/kaiju2table.sh
